In [ ]:
# Cell 1: 환경 설정
!pip install -q transformers==4.36.2 datasets accelerate peft bitsandbytes
!pip install -q wandb sentencepiece

In [ ]:
!pip uninstall -y transformers accelerate
!pip install transformers==4.36.2 accelerate==0.21.0

In [ ]:
import json
import torch

# GPU 확인
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"GPU: {gpu_name}")

# 수정된 설정 - evaluation_strategy를 "no"로 변경
config = {
    # 모델 설정
    "model_name_or_path": "./KoAlpaca-Polyglot-5.8B",
    "trust_remote_code": True,
    
    # 데이터 설정
    "train_data_path": "/content/all_faq.jsonl",
    "eval_data_path": None,  # 평가 데이터 없음
    "max_length": 1024,
    "prompt_template_name": "korean",
    
    # QLoRA 설정
    "lora_r": 32,
    "lora_alpha": 64,
    "lora_dropout": 0.1,
    "bnb_4bit_compute_dtype": "float16",
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_use_double_quant": True,
    
    # 학습 설정
    "output_dir": "./koalpaca-qlora-output",
    "num_train_epochs": 3,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "gradient_checkpointing": True,
    "optim": "paged_adamw_8bit",
    "learning_rate": 8e-5,
    "warmup_ratio": 0.03,
    "max_grad_norm": 0.3,
    "lr_scheduler_type": "cosine",
    
    # 중요: 평가 전략을 "no"로 설정
    "evaluation_strategy": "no",  # ✅ 평가 비활성화
    "save_strategy": "steps",      # epoch 대신 steps로 변경
    "save_steps": 100,             # 100 스텝마다 저장
    "logging_steps": 10,
    "fp16": True,
    "report_to": ["none"],
}

# L4 GPU인 경우 더 큰 배치 가능
if "L4" in gpu_name:
    config["per_device_train_batch_size"] = 2
    config["gradient_accumulation_steps"] = 4
    config["lora_r"] = 64

# 설정 저장
with open("fixed_qlora_config.json", "w") as f:
    json.dump(config, f, indent=2)

print("✅ 수정된 설정 파일 생성 완료!")

In [ ]:
# Cell 2: 간단한 학습 스크립트 (오류 수정 버전)
%%writefile train_qlora_fixed.py
import os
import sys
import json
import torch
import logging
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# 환경 변수 설정
os.environ["TOKENIZERS_PARALLELISM"] = "false"

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def load_jsonl(file_path):
    """JSONL 파일 로드"""
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

def format_prompt(example):
    """프롬프트 포맷팅"""
    instruction = example.get('instruction', '')
    input_text = example.get('input', '')
    output = example.get('output', '')
    
    if input_text:
        prompt = f"""### 명령어:
{instruction}

### 입력:
{input_text}

### 응답:
{output}"""
    else:
        prompt = f"""### 명령어:
{instruction}

### 응답:
{output}"""
    return prompt

def main():
    # 설정 로드
    with open(sys.argv[1], 'r') as f:
        config = json.load(f)
    
    logger.info("설정 로드 완료")
    
    # 4bit 설정
    compute_dtype = getattr(torch, config.get("bnb_4bit_compute_dtype", "float16"))
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type=config.get("bnb_4bit_quant_type", "nf4"),
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=config.get("bnb_4bit_use_double_quant", True),
    )
    
    # 모델 로드
    logger.info("모델 로드 중...")
    model = AutoModelForCausalLM.from_pretrained(
        config["model_name_or_path"],
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=config.get("trust_remote_code", True),
        use_cache=False,  # gradient checkpointing과 호환되도록
    )
    
    # gradient checkpointing 활성화
    model.gradient_checkpointing_enable()
    model.config.use_cache = False  # 명시적으로 비활성화
    
    # 토크나이저 로드
    tokenizer = AutoTokenizer.from_pretrained(
        config["model_name_or_path"],
        trust_remote_code=config.get("trust_remote_code", True),
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # PEFT 설정
    model = prepare_model_for_kbit_training(model)
    
    # Polyglot 모델용 타겟 모듈
    target_modules = [
        "attention.query_key_value",
        "attention.dense",
        "mlp.dense_h_to_4h",
        "mlp.dense_4h_to_h"
    ]
    
    lora_config = LoraConfig(
        r=config.get("lora_r", 32),
        lora_alpha=config.get("lora_alpha", 64),
        target_modules=target_modules,
        lora_dropout=config.get("lora_dropout", 0.1),
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )
    
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    
    # 데이터 준비
    logger.info("데이터 준비 중...")
    train_data = load_jsonl(config["train_data_path"])
    
    # 프롬프트 포맷팅
    formatted_data = []
    for item in train_data:
        formatted_data.append({"text": format_prompt(item)})
    
    # 데이터셋 생성
    dataset = Dataset.from_list(formatted_data)
    
    # 토큰화 함수
    def tokenize_function(examples):
        outputs = tokenizer(
            examples["text"],
            truncation=True,
            padding="max_length",
            max_length=config.get("max_length", 1024),
        )
        outputs["labels"] = outputs["input_ids"].copy()
        return outputs
    
    # 토큰화
    tokenized_dataset = dataset.map(
        tokenize_function,
        batched=True,
        remove_columns=["text"],
    )
    
    # 학습 인자 설정
    training_args = TrainingArguments(
        output_dir=config["output_dir"],
        num_train_epochs=config.get("num_train_epochs", 3),
        per_device_train_batch_size=config.get("per_device_train_batch_size", 1),
        gradient_accumulation_steps=config.get("gradient_accumulation_steps", 8),
        warmup_ratio=config.get("warmup_ratio", 0.03),
        learning_rate=config.get("learning_rate", 0.0002),
        fp16=config.get("fp16", True),
        logging_steps=config.get("logging_steps", 10),
        save_strategy=config.get("save_strategy", "steps"),
        save_steps=config.get("save_steps", 100),
        evaluation_strategy=config.get("evaluation_strategy", "no"),  # 평가 비활성화
        optim=config.get("optim", "paged_adamw_8bit"),
        max_grad_norm=config.get("max_grad_norm", 0.3),
        lr_scheduler_type=config.get("lr_scheduler_type", "cosine"),
        report_to=config.get("report_to", ["none"]),
        ddp_find_unused_parameters=False,  # DDP 오류 방지
    )
    
    # Trainer 생성
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset,
        tokenizer=tokenizer,
        data_collator=DataCollatorForLanguageModeling(
            tokenizer=tokenizer,
            mlm=False,
        ),
    )
    
    # 학습
    logger.info("학습 시작...")
    trainer.train()
    
    # 모델 저장
    logger.info("모델 저장 중...")
    trainer.save_model()
    tokenizer.save_pretrained(config["output_dir"])
    
    logger.info(f"완료! 모델 저장 위치: {config['output_dir']}")

if __name__ == "__main__":
    main()


In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

# 수정된 스크립트로 학습 실행
!python train_qlora_fixed.py fixed_qlora_config.json

In [ ]:
# Cell 1: 기존 파인튜닝 모델 확인
import os
import json
import torch
from transformers import AutoTokenizer
from peft import PeftModel, PeftConfig

def check_existing_model(adapter_path="./koalpaca-qlora-output"):
    """기존 파인튜닝 모델 정보 확인"""
    
    print("🔍 기존 파인튜닝 모델 확인")
    print("="*60)
    
    # adapter_config.json 확인
    config_path = os.path.join(adapter_path, "adapter_config.json")
    if os.path.exists(config_path):
        with open(config_path, 'r') as f:
            adapter_config = json.load(f)
        
        print("✅ LoRA 설정:")
        print(f"  - r (rank): {adapter_config.get('r', 'N/A')}")
        print(f"  - alpha: {adapter_config.get('lora_alpha', 'N/A')}")
        print(f"  - dropout: {adapter_config.get('lora_dropout', 'N/A')}")
        print(f"  - target_modules: {adapter_config.get('target_modules', 'N/A')}")
    
    # 모델 파일 크기 확인
    model_files = [f for f in os.listdir(adapter_path) if f.endswith('.bin') or f.endswith('.safetensors')]
    total_size = sum(os.path.getsize(os.path.join(adapter_path, f)) for f in model_files) / (1024**2)
    print(f"\n✅ 어댑터 크기: {total_size:.1f} MB")
    
    return adapter_config

# 기존 모델 확인
existing_config = check_existing_model()

In [ ]:
# Cell 2: 추가 파인튜닝을 위한 설정
def create_continual_finetuning_config(
    existing_adapter_path="./koalpaca-qlora-output",
    new_data_path="/content/all_faq.jsonl",
    output_dir="./koalpaca-continual-ft"
):
    """추가 파인튜닝 설정 생성"""
    
    # 기존 어댑터 설정 로드
    with open(os.path.join(existing_adapter_path, "adapter_config.json"), 'r') as f:
        adapter_config = json.load(f)
    
    # GPU 확인
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    print(f"GPU: {gpu_name}")
    
    config = {
        # 모델 설정 - 기존과 동일한 베이스 모델 사용
        "model_name_or_path": adapter_config.get("base_model_name_or_path", "beomi/KoAlpaca-Polyglot-5.8B"),
        "existing_adapter_path": existing_adapter_path,  # ✅ 기존 어댑터 경로
        "trust_remote_code": True,
        
        # 데이터 설정 - 새로운 데이터
        "train_data_path": new_data_path,
        "max_length": 1024,
        "prompt_template_name": "korean",
        
        # QLoRA 설정 - 기존과 동일하게 유지
        "lora_r": adapter_config.get("r", 32),
        "lora_alpha": adapter_config.get("lora_alpha", 64),
        "lora_dropout": adapter_config.get("lora_dropout", 0.1),
        "bnb_4bit_compute_dtype": "float16",
        "bnb_4bit_quant_type": "nf4",
        "bnb_4bit_use_double_quant": True,
        
        # 학습 설정 - 더 낮은 학습률 사용
        "output_dir": output_dir,
        "num_train_epochs": 3,  # 적은 epoch
        "per_device_train_batch_size": 1,
        "gradient_accumulation_steps": 8,
        "gradient_checkpointing": True,
        "optim": "paged_adamw_8bit",
        "learning_rate": 8e-5,  # ✅ 더 낮은 학습률
        "warmup_ratio": 0.1,  # 더 긴 warmup
        "max_grad_norm": 0.3,
        "lr_scheduler_type": "cosine",
        "evaluation_strategy": "no",
        "save_strategy": "steps",
        "save_steps": 50,
        "logging_steps": 10,
        "fp16": True,
        "report_to": ["none"],
    }
    
    # 설정 저장
    with open("continual_ft_config.json", "w") as f:
        json.dump(config, f, indent=2)
    
    print(f"\n✅ 추가 파인튜닝 설정 생성 완료!")
    print(f"  - 기존 모델: {existing_adapter_path}")
    print(f"  - 새 데이터: {new_data_path}")
    print(f"  - 출력 경로: {output_dir}")
    
    return config

# 설정 생성
continual_config = create_continual_finetuning_config()


In [ ]:
# Cell 4: 추가 파인튜닝 스크립트
%%writefile train_continual_ft.py
import os
import sys
import json
import torch
import logging
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from datasets import Dataset
from peft import (
    PeftModel,
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType
)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def load_jsonl(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

def format_prompt(example):
    instruction = example.get('instruction', '')
    input_text = example.get('input', '')
    output = example.get('output', '')
    
    if input_text:
        prompt = f"""### 명령어:
{instruction}

### 입력:
{input_text}

### 응답:
{output}"""
    else:
        prompt = f"""### 명령어:
{instruction}

### 응답:
{output}"""
    return prompt

def main():
    # 설정 로드
    with open(sys.argv[1], 'r') as f:
        config = json.load(f)
    
    logger.info("추가 파인튜닝 시작")
    
    # 4bit 설정
    compute_dtype = getattr(torch, config.get("bnb_4bit_compute_dtype", "float16"))
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type=config.get("bnb_4bit_quant_type", "nf4"),
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=config.get("bnb_4bit_use_double_quant", True),
    )
    
    # 베이스 모델 로드
    logger.info("베이스 모델 로드 중...")
    base_model = AutoModelForCausalLM.from_pretrained(
        config["model_name_or_path"],
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=config.get("trust_remote_code", True),
        use_cache=False,
    )

    base_model = prepare_model_for_kbit_training(base_model)
    
    # 기존 어댑터 로드
    logger.info(f"기존 어댑터 로드 중: {config['existing_adapter_path']}")
    model = PeftModel.from_pretrained(
        base_model,
        config["existing_adapter_path"],
        is_trainable=True  # ✅ 학습 가능하도록 설정
    )
    
    # gradient checkpointing 활성화
    model.gradient_checkpointing_enable()
    model.config.use_cache = False
    
    # 학습 모드로 전환
    model.train()
    
    # 토크나이저 로드
    tokenizer = AutoTokenizer.from_pretrained(
        config["model_name_or_path"],
        trust_remote_code=config.get("trust_remote_code", True),
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # 학습 가능한 파라미터 확인
    model.print_trainable_parameters()
    
    # 데이터 준비
    logger.info("새로운 데이터 준비 중...")
    train_data = load_jsonl(config["train_data_path"])
    
    formatted_data = []
    for item in train_data:
        formatted_data.append({"text": format_prompt(item)})
    
    dataset = Dataset.from_list(formatted_data)
    
    def tokenize_function(examples):
        outputs = tokenizer(
            examples["text"],
            truncation=True,
            padding="max_length",
            max_length=config.get("max_length", 1024),
        )
        outputs["labels"] = outputs["input_ids"].copy()
        return outputs
    
    tokenized_dataset = dataset.map(
        tokenize_function,
        batched=True,
        remove_columns=["text"],
    )
    
    # 학습 인자 설정
    training_args = TrainingArguments(
        output_dir=config["output_dir"],
        num_train_epochs=config.get("num_train_epochs", 3),
        per_device_train_batch_size=config.get("per_device_train_batch_size", 1),
        gradient_accumulation_steps=config.get("gradient_accumulation_steps", 8),
        warmup_ratio=config.get("warmup_ratio", 0.1),
        learning_rate=config.get("learning_rate", 8e-5,),  # 낮은 학습률
        fp16=config.get("fp16", True),
        logging_steps=config.get("logging_steps", 10),
        save_strategy=config.get("save_strategy", "steps"),
        save_steps=config.get("save_steps", 50),
        evaluation_strategy=config.get("evaluation_strategy", "no"),
        optim=config.get("optim", "paged_adamw_8bit"),
        max_grad_norm=config.get("max_grad_norm", 0.3),
        lr_scheduler_type=config.get("lr_scheduler_type", "cosine"),
        report_to=config.get("report_to", ["none"]),
        ddp_find_unused_parameters=False,
    )
    
    # Trainer 생성
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset,
        tokenizer=tokenizer,
        data_collator=DataCollatorForLanguageModeling(
            tokenizer=tokenizer,
            mlm=False,
        ),
    )
    
    # 학습
    logger.info("추가 파인튜닝 시작...")
    trainer.train()
    
    # 모델 저장
    logger.info("업데이트된 모델 저장 중...")
    trainer.save_model()
    tokenizer.save_pretrained(config["output_dir"])
    
    logger.info(f"완료! 업데이트된 모델 저장 위치: {config['output_dir']}")

if __name__ == "__main__":
    main()


In [ ]:
# Cell 5: 추가 파인튜닝 실행
import gc
gc.collect()
torch.cuda.empty_cache()

# 추가 파인튜닝 실행
!python train_continual_ft.py continual_ft_config.json

In [ ]:
# Cell 6: 기존 지식 유지 확인 테스트
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

def test_knowledge_retention(
    original_adapter="./koalpaca-qlora-output",
    updated_adapter="./koalpaca-continual-ft"
):
    """기존 지식이 유지되는지 테스트"""
    
    # 4bit 설정
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    
    # 베이스 모델 로드
    base_model = AutoModelForCausalLM.from_pretrained(
        "beomi/KoAlpaca-Polyglot-5.8B",
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    
    tokenizer = AutoTokenizer.from_pretrained("beomi/KoAlpaca-Polyglot-5.8B")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # 테스트 질문들
    test_questions = [
        # 기존 지식 테스트
        "신용카드 부정사용 시 대응 방법을 알려주세요.",
        "보험금 지급 거절 시 소비자의 권리는?",
        # 새로운 지식 테스트
        "가상자산 투자 사기를 당했을 때 대응 방법은?",
        "마이데이터 개인정보 유출 시 어떻게 해야 하나요?",
    ]
    
    def generate_response(model, question):
        prompt = f"""### 명령어:
{question}

### 응답:
"""
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=512,
            return_token_type_ids=False
        )
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs.to(model.device),
                max_new_tokens=256,
                temperature=0.7,
                do_sample=True,
                pad_token_id=tokenizer.pad_token_id,
            )
        
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        return response.split("### 응답:")[-1].strip()
    
    print("="*60)
    print("🔍 지식 유지 테스트")
    print("="*60)
    
    # 원본 모델 테스트
    print("\n[ 원본 파인튜닝 모델 ]")
    original_model = PeftModel.from_pretrained(base_model, original_adapter)
    original_model.eval()
    
    for i, question in enumerate(test_questions[:2]):  # 기존 지식만
        print(f"\nQ{i+1}: {question}")
        print(f"A: {generate_response(original_model, question)[:200]}...")
    
    # 메모리 정리
    del original_model
    gc.collect()
    torch.cuda.empty_cache()
    
    # 업데이트된 모델 테스트
    print("\n\n[ 추가 파인튜닝된 모델 ]")
    base_model = AutoModelForCausalLM.from_pretrained(
        "beomi/KoAlpaca-Polyglot-5.8B",
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    updated_model = PeftModel.from_pretrained(base_model, updated_adapter)
    updated_model.eval()
    
    for i, question in enumerate(test_questions):  # 모든 질문
        print(f"\nQ{i+1}: {question}")
        print(f"A: {generate_response(updated_model, question)[:200]}...")
    
    print("\n✅ 테스트 완료!")

# 테스트 실행
test_knowledge_retention()